# LOF And Isolation Forest

Which observations deserve inspection, and why? The slides developed two comparisons: local density relative to neighbors, and ease of isolation by random partitions. Here we work through the LOF calculations, apply both methods to the same seven observations, and test choices that affect their interpretation.

Use **Apply LOF And Inspect Local Support** after the LOF slides. Return to **Isolation Through Random Partitions** in the deck before continuing to **Apply Isolation Forest And Compare Evidence** below. After comparing methods, return to the slides at **Scores, Flags, And Investigation**, then use the matching notebook section to apply selection rules. Run the notebook in order; later sections use the observations and scores from earlier sections.

## References

- [Scikit-learn anomaly-detection guide](https://scikit-learn.org/stable/modules/outlier_detection.html): explanations of the methods and reference situations.
- [Local Outlier Factor API](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.LocalOutlierFactor.html): fitted-collection scores and the separate novelty interface.
- [Isolation Forest API](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html): tree settings and score conventions.

## Apply LOF And Inspect Local Support

LOF compares local reachability densities. The slides established the sequence: adjust neighbor distances, average and invert those distances, then compare neighbors' average density with the observation's own density.

Work through the numbers with k = 2 before fitting the software. We will trace 0 to see the short-distance adjustment, then calculate and compare LOF for 10 and 80.

### Reachability Distances

![Seven observations on one continuous number line: 0, 1, and 3 form a tight group; 50, 60, and 80 form a wider group; 10 lies beyond the tight group.](figures/neighbor-number-line.png)

Use the same seven values: `0, 1, 3, 10, 50, 60, 80`. With k = 2, each observation's radius is its distance to the second-nearest other observation.

| Observation | Two Nearest Others | Second-Neighbor Radius |
| --- | --- | --- |
| 0 | 1, 3 | 3 |
| 1 | 0, 3 | 2 |
| 3 | 1, 0 | 3 |
| 10 | 3, 1 | 9 |
| 50 | 60, 80 | 30 |
| 60 | 50, 80 | 20 |
| 80 | 60, 50 | 30 |

Let p be the observation being scored and o one of its neighbors. The ordinary distance between them is d(p, o). Write $r_k(o)$ for the distance from o to **its own kth-nearest other observation**, excluding o itself. This is o's neighborhood radius.

The table's Second-Neighbor Radius column gives $r_2(o)$. For example, observation 1 has neighbors 0 and 3 at distances 1 and 2, so $r_2(1)=2$.

Reachability takes the larger of actual separation and the **receiving neighbor's radius**. This uses the neighbor's surrounding scale as a lower bound on short distances.

$$
\operatorname{reach}_k(p,o)=\max\{d(p,o),r_k(o)\}
$$

For observation 0:

- To neighbor 1: actual distance 1; neighbor radius 2.
  - Adjusted distance = max(1, 2) = 2.
- To neighbor 3: actual distance 3; neighbor radius 3.
  - Adjusted distance = max(3, 3) = 3.

The short contribution increases while the larger one stays. Moving closer to a neighbor within its fixed radius would not reduce the contribution further.

For observation 10, both actual gaps already exceed the neighbors' radii:

$$
\operatorname{reach}_2(10,3)=\max(7,3)=7
$$

$$
\operatorname{reach}_2(10,1)=\max(9,2)=9
$$

Both gaps stay. The adjustment moderates small spacing differences within a neighborhood while retaining larger gaps.

### Direction In Reachability

From 0 to 1, the adjusted distance was max(1, 2) = 2. Reverse the comparison: from 1 to 0, which observation supplies the radius? What is the adjusted distance?

##### Answer

Observation 0 now supplies its radius of 3. The adjusted distance is max(1, 3) = 3. Ordinary separation is symmetric, but reachability uses the receiving neighbor's context. The two directions can therefore have different adjusted distances.

### Local Reachability Densities

The **local reachability density**, written $\operatorname{lrd}_k(p)$, measures local packing around observation p using its k-neighbor comparison. Average the reachability distances from p to its neighbors, then take the reciprocal:

$$
\operatorname{lrd}_k(p)=\frac{1}{\text{mean reachability distance from p to its neighbors}}
$$

Smaller mean reachability distance gives higher local reachability density. For 0, the adjusted distances are 2 and 3:

$$
\operatorname{lrd}_2(0)=\frac{1}{(2+3)/2}=0.4
$$

The preliminary inverse mean distance in the slides was .5. Raising the short-distance contribution reduces this density estimate to .4.

For 10, the distances remain 7 and 9:

$$
\operatorname{lrd}_2(10)=\frac{1}{(7+9)/2}=0.125
$$

Now calculate the same quantity for 10's neighbors, 3 and 1:

- For 3, neighbors are 1 and 0.
  - Adjusted distances: max(2, 2) = 2 and max(3, 3) = 3.
  - Mean = 2.5; local reachability density = .4.
- For 1, neighbors are 0 and 3.
  - Adjusted distances: max(1, 3) = 3 and max(2, 3) = 3.
  - Mean = 3; local reachability density = 1/3.

We now have the densities needed to compare 10 with its neighbors.

### LOF For Observation 10

The **Local Outlier Factor**, written $\operatorname{LOF}_k(p)$, compares the average local reachability density of p's neighbors with p's own local reachability density:

$$
\operatorname{LOF}_k(p)=\frac{\text{mean local reachability density of p's neighbors}}{\operatorname{lrd}_k(p)}
$$

Both numerator and denominator use the same neighborhood size k. A ratio near 1 indicates comparable local densities; a ratio above 1 means p has lower density than its neighbors' average.

For observation 10, divide its neighbors' average local reachability density by its own density:

$$
\operatorname{LOF}_2(10)=\frac{(0.4+1/3)/2}{0.125}\approx2.93
$$

The numerator is about .3667; the denominator is .125. The neighbors' average density is almost three times 10's own density. Higher LOF indicates weaker local support relative to neighbors; near 1 indicates comparable local densities.

### Supporting LOF Calculations

For reference, these are the adjusted distances used with k = 2. Each row takes the larger of actual separation and the receiving neighbor's radius, averages the two adjusted distances, and takes the reciprocal. The ratio then compares the neighbors' average density with the row's own density.

| Observation | Adjusted Distances | Mean | Local Reachability Density |
| --- | --- | --- | --- |
| 0 | max(1, 2) = 2; max(3, 3) = 3 | 2.5 | .4000 |
| 1 | max(1, 3) = 3; max(2, 3) = 3 | 3 | .3333 |
| 3 | max(2, 2) = 2; max(3, 3) = 3 | 2.5 | .4000 |
| 10 | max(7, 3) = 7; max(9, 2) = 9 | 8 | .1250 |
| 50 | max(10, 20) = 20; max(30, 30) = 30 | 25 | .0400 |
| 60 | max(10, 30) = 30; max(20, 30) = 30 | 30 | .0333 |
| 80 | max(20, 20) = 20; max(30, 30) = 30 | 25 | .0400 |

Values are rounded for display. The calculations and fitted scores use unrounded values.

### Local Comparison For 80

From the table, 80's mean adjusted distance is 25, giving density .04. Its neighbors are 60 and 50, with densities 1/30 and .04:

$$
\operatorname{LOF}_2(80)=\frac{(1/30+0.04)/2}{0.04}\approx0.92
$$

Observation 80 has lower absolute density than 10, but its density is comparable to its own neighbors. The local comparison distinguishes unusual spacing within a region from simply belonging to a loosely packed region.

### LOF Results

The plot shows the scores obtained by repeating these calculations for all seven observations. Horizontal positions retain the original feature values; height shows LOF. Labels identify the observations, and the dashed line at 1 marks equal density relative to the neighbor average.

![LOF at k equal to two, against original feature value](figures/lof-scores.png)

Observation 10 stands above the others. Observations 1 and 60 score 1.20 because their adjusted neighborhood geometry gives them slightly weaker support than their neighbors. We will reproduce the scores in software, then change k to see how the reference neighborhoods affect the comparison.

### Observations And Input

Each row contains one observation of a single numerical feature. These invented values make the geometry and arithmetic inspectable: 0, 1, and 3 form the tight group; 50, 60, and 80 form the wide group; 10 lies beyond the tight group. We identify observations by their feature values throughout this example.

The following cell imports the tools and records the installed scikit-learn version. The calculations were developed with version 1.9.0.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

print(f"scikit-learn {sklearn.__version__}")

Create the input table. Scikit-learn expects rows for observations and columns for features, even with only one feature. Selecting `[["value"]]` keeps a two-dimensional DataFrame with seven rows and one column.

In [ ]:
records = pd.DataFrame({"value": [0, 1, 3, 10, 50, 60, 80]})
X = records[["value"]]
display(X)
print(f"Input shape: {X.shape}")

### Fit And Inspect LOF

- Construct an estimator with `n_neighbors=2` to match the hand calculation.
  - The library default is 20; two is our deliberate choice for this example.
  - `metric="euclidean"` makes the distance rule explicit. In one dimension this is absolute difference.
- Call `fit(X)` to establish neighborhoods and calculate scores for these observations.
  - Each observation is excluded from its own neighbor set.
- Read `negative_outlier_factor_`, a learned attribute of the fitted estimator.
  - Scikit-learn stores the negative of mathematical LOF. Negate it so larger means more unusual.

In [ ]:
lof_model = LocalOutlierFactor(n_neighbors=2, metric="euclidean")
lof_model.fit(X)
lof_scores = -lof_model.negative_outlier_factor_

results = records.copy()
results["LOF k=2"] = lof_scores
display(results.round(3))

Each row still refers to the same original observation. Observation 10 scores 2.933; observation 80 scores .917. These reproduce the hand calculations. Observations 1 and 60 score 1.200 because their adjusted neighborhood geometry gives them slightly lower density than their neighbors.

A score near 1 means comparable local density. Selecting observations for investigation requires an additional rule, which we will apply after comparing the methods.

### Inspect The Neighbors

The fitted model can return the neighbors used for each training observation. Calling `kneighbors()` with no new input requests the fitted observations' neighborhoods and excludes each observation itself. It returns two arrays: neighbor distances and row positions. Use the row positions to retrieve the original feature values.

In [ ]:
neighbor_distances, neighbor_rows = lof_model.kneighbors()
values = records["value"].to_numpy()

neighborhoods = records.copy()
neighborhoods["neighbor 1"] = values[neighbor_rows[:, 0]]
neighborhoods["distance 1"] = neighbor_distances[:, 0]
neighborhoods["neighbor 2"] = values[neighbor_rows[:, 1]]
neighborhoods["distance 2"] = neighbor_distances[:, 1]
display(neighborhoods)

Read the row for 10: neighbors 3 and 1, at distances 7 and 9. The row for 80 gives neighbors 60 and 50, at distances 20 and 30. The returned distances are ordinary Euclidean distances; the reachability adjustment occurs inside the LOF calculation.

### Predict A Neighborhood Change

Keep all seven values fixed and increase k from 2 to 3. Each three-point group must now include an observation outside itself.

- Which new neighbor must 50 include after 60 and 80?
- Will the wide group still supply all of its own reference observations?
- Predict how this could change the comparison between 10 and the wide group.

##### Answer

Observation 50 must include 10, at distance 40. The wide group's neighborhoods now extend toward the tight region. Meanwhile, the tight group can include 10 as its additional neighbor. The regions defining local support change, so the density comparisons can change substantially. Exact scores require recalculating the adjusted distances and densities as well as the neighbor lists.

### Refit With Three Neighbors

Create a second fitted estimator so we retain the original result for comparison. The new fit recomputes neighborhoods, reachability distances, local densities, and the final ratios. Add its scores to the same observation table.

In [ ]:
lof_three = LocalOutlierFactor(n_neighbors=3, metric="euclidean")
lof_three.fit(X)
results["LOF k=3"] = -lof_three.negative_outlier_factor_
display(results.round(3))

Observation 10 falls from 2.933 to .953. The wide-group observations rise to about 2.718. Increasing k changed which region supplies the reference for local density.

The next cell plots both sets of scores against the original feature values. Both panels keep the same axes; the vertical movement shows the score change. Point labels identify the observations, and the line at 1 marks equal density relative to the neighbor average.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), layout="constrained")
offsets = [(-15, -24), (0, 15), (16, -24), (0, 12), (0, 12), (0, 12), (0, 12)]
for ax, k in zip(axes, [2, 3], strict=True):
    scores = results[f"LOF k={k}"]
    ax.axhline(1, color="#697582", linestyle="--", linewidth=1)
    ax.scatter(values, scores, s=60, color="#245980", zorder=3)
    for value, score, offset in zip(values, scores, offsets, strict=True):
        ax.annotate(str(value), (value, score), xytext=offset,
                    textcoords="offset points", ha="center", fontsize=11,
                    arrowprops={"arrowstyle": "-", "color": "#697582"})
    ax.set(xlim=(-5, 85), ylim=(.3, 3.4), ylabel="LOF", title=f"LOF with k = {k}",
           xlabel="Original feature value")
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

The high scores move from 10 to the wide group even though the observations stay fixed. Neighborhood size sets the scale of the comparison. A useful k depends on which local structure matters to the task.

### Return To Slides

LOF compares local support, and we have seen how changing the neighborhood changes that comparison. Could another criterion prioritize different observations in the same collection?

Return to **Isolation Through Random Partitions** in the slides. Continue below after the explanation of splits, paths, forests, and changes of units.

## Apply Isolation Forest And Compare Evidence

Isolation Forest scores how easily random partitions separate each observation. Short paths across many trees indicate easy isolation. We will fit the forest, inspect its scores beside LOF, and explain any disagreement using the same seven values.

### Fit The Forest

The constructor specifies how the forest will be built; `fit(X)` builds the trees using the observations.

- `n_estimators=1000`: build 1,000 trees instead of the default 100.
  - This stabilizes the small demonstration across many random partition sequences.
- `max_samples=len(X)`: use all seven observations to build every tree.
  - Hold the observations fixed while the random partitions vary.
- `random_state=0`: fix the random seed so the example can be reproduced.

No fault labels are supplied. After fitting, `score_samples(X)` routes the observations through the trees and returns one score per row. Scikit-learn gives lower scores to easier-to-isolate observations; negate them to use larger-means-more-unusual throughout our comparison.

In [ ]:
forest = IsolationForest(n_estimators=1000, max_samples=len(X), random_state=0)
forest.fit(X)
results["isolation score"] = -forest.score_samples(X)
display(results[["value", "isolation score"]].round(3))

Observation 80 has the highest score, about .630; 10 scores about .522. The endpoint at 80 has a large adjacent gap. Many random cuts separate it quickly, giving short paths across the forest.

The reported score uses the sample-size reference discussed in the slides. It measures ease of isolation on that scale; .630 is not a 63% probability of fault.

### Compare Detector Scores

Put LOF at k = 2 and the isolation score beside each observation. Both columns use larger-means-more-unusual, but their numerical scales express different quantities. Compare which observations each method prioritizes.

In [ ]:
comparison = results[["value", "LOF k=2", "isolation score"]]
display(comparison.round(3))

Plot one point per observation using its two scores as coordinates. Horizontal position shows LOF; vertical position shows the isolation score. Labels give the original feature value, so we can trace a score back to the observation that produced it.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5), layout="constrained")
ax.scatter(results["LOF k=2"], results["isolation score"], s=70, color="#245980")
offsets = [(-25, 12), (10, -2), (-25, -17), (-16, 10), (-25, 8), (12, -10), (12, 0)]
for value, lof, isolation, offset in zip(
    values, results["LOF k=2"], results["isolation score"], offsets, strict=True
):
    ax.annotate(str(value), (lof, isolation), xytext=offset, textcoords="offset points",
                fontsize=12, arrowprops={"arrowstyle": "-", "color": "#697582"})
ax.set(xlim=(.65, 3.2), ylim=(.39, .66),
       xlabel="LOF (k = 2): more unusual →", ylabel="Isolation score: more unusual →",
       title="Detector scores for the same seven observations")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

- Observation 10 lies farthest right: highest LOF.
- Observation 80 lies highest: easiest isolation across the forest.
- Observations 0, 1, and 3 are relatively low under both criteria.

The axes retain their own scales. A diagonal would not express equal judgments by the two methods.

### Explain The Disagreement

Why does LOF place 10 first while Isolation Forest places 80 first? Use their neighbors and positions in the original collection to explain both results. What additional information would help decide which deserves investigation in a pump application?

##### Answer

LOF compares local densities. Observation 10 has weaker support than its tight-group neighbors. Observation 80's neighbors are also widely spaced, so its density is comparable to theirs.

Isolation Forest measures ease of separation. Observation 80 is an endpoint with a large adjacent gap, making it easy to isolate under many random cuts.

The useful criterion depends on the task. A known, legitimate high-load regime may explain a wide group of readings; a physically concerning endpoint may still deserve inspection. Check operating conditions, sensor validity, and evidence of actual faults before assigning operational meaning to the scores.

### Change The Random Seed

Does the ordering between 80 and 10 depend on one set of random draws? Fit another forest with seed 1 while holding the data, number of trees, and per-tree sample size fixed. Compare each observation with itself across the two score columns.

In [ ]:
second_forest = IsolationForest(n_estimators=1000, max_samples=len(X), random_state=1)
second_forest.fit(X)
seed_comparison = records.copy()
seed_comparison["seed 0"] = results["isolation score"]
seed_comparison["seed 1"] = -second_forest.score_samples(X)
display(seed_comparison.round(3))

The exact scores change slightly. Observation 80 remains above 10: about .628 versus .526 under seed 1. This comparison supports repeatability under these two draws. It does not tell us whether either observation represents a fault.

### Return To Slides

We have compared scores from both detectors. Return to **Scores, Flags, And Investigation** in the slides to establish selection rules and the distinction between a flag and an actual fault. Then continue with the matching section below.

## Scores, Flags, And Investigation

The slides established the chain: score, selection rule, flag, investigation, and evaluation against the actual condition. Here we apply rules to the existing LOF scores. In a pump application, the selected readings might receive a sensor check or maintenance investigation.

### Apply Selection Rules

Use LOF greater than 2, then lower the threshold to 1.1. The values make the change in selected observations inspectable. Operational thresholds require evidence about the target condition and consequences; an isolation-score threshold would use that method's own scale.

Before running the cell, use the displayed LOF scores to predict which observations each rule will flag.

In [ ]:
flags = results[["value", "LOF k=2"]].copy()
flags["LOF > 2"] = flags["LOF k=2"] > 2
flags["LOF > 1.1"] = flags["LOF k=2"] > 1.1
display(flags.round(3))

The comparison `>` creates a Boolean value for each row: `True` means the row meets the selection rule. The first rule flags 10. Lowering the threshold adds 1 and 60 because both score 1.20. The scores, neighborhoods, and rankings stay unchanged.

Changing k changed the evidence used to score an observation. Changing the threshold selects a different set from existing scores.

### Inspect Flagged Observations

Use the Boolean column to select the original observations and their scores. In a pump dataset, keep identifying fields and operating measurements beside these results so the selected readings can be investigated.

In [ ]:
selected = flags.loc[flags["LOF > 1.1"], ["value", "LOF k=2"]]
display(selected.round(3))

Other selection rules can use a review capacity or combine a score with operating criteria. State the rule and the follow-up it requests. The analytical chain is: calculate scores, apply the rule, inspect selected observations, and seek evidence about their condition.

Scikit-learn can also produce binary labels using its own cutoff. A numerical `contamination` setting supplies a fraction used to choose a training-score cutoff. That choice does not establish the fraction of actual faults. Our explicit comparisons above expose the rule directly.

### False Alarms And Missed Faults

Use the slide definitions: positive/negative describes whether a flag was raised; true/false describes whether that decision matches independently established condition.

Imagine ten readings whose actual conditions have been established. Two involve faults. The detector flags three readings: one faulty and two healthy.

How many true positives, false positives, false negatives, and true negatives are there? Explain why counting three flags cannot tell us how many faults were found.

##### Answer

- One fault flagged: one true positive.
- Two healthy readings flagged: two false positives.
- One fault missed: one false negative.
- Six healthy readings not flagged: six true negatives.

The three flags include only one fault. Counting flags measures the amount selected for investigation. Evaluating those decisions requires the actual conditions, including evidence about unflagged readings if we want to count missed faults.

### Interpreting Flags

Compare the two threshold selections above. Which observations would require additional investigation after lowering the threshold, and what would we need to learn about them to judge whether that change was useful?

Explain why one detector prioritizes its leading observation. Name one additional check that would help determine whether an analogous pump reading deserves action.

## Scoring New Observations

The calculations above assess observations within one collection. For a healthy reference collection followed by new readings, scikit-learn calls the task **novelty detection**. The reference data and the observations being assessed have different roles.

LOF uses a separate interface for that task: set `novelty=True` before fitting the reference collection, then score new observations. Those novelty calls should be used for new observations; applying them to the training collection does not reproduce the fitted-collection scores used here. Isolation Forest can route new observations through its fitted trees using `score_samples`.

In either case, choose a reference that represents the operating conditions relevant to the investigation.